# 01 — Data Preprocessing

## Multi-Agent Financial Analysis System / SignalScout

This notebook creates the standardized preprocessing layer for:
1. SEC Form 4 insider transactions
2. Market data
3. Financial news

Raw data stays unchanged. Cleaned outputs are written to `data/processed/` and reused by later agent notebooks.


## 1. Imports and Shared Configuration

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from shared import PATHS, SEED, ensure_directories, set_seed  # noqa: E402

set_seed()
ensure_directories()

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Project root:", PROJECT_ROOT)
print("Seed:", SEED)
print("Processed path:", PATHS["processed"])

Project root: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System
Seed: 42
Processed path: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed


## 2. Shared Preprocessing Helpers

In [2]:
def normalize_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df

def normalize_ticker(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA})
    )

def clean_text(series):
    return (
        series.astype("string")
        .str.replace(r"<[^>]+>", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def to_utc(series):
    return pd.to_datetime(series, errors="coerce", utc=True)

def report_rows(name, before, after):
    print(f"{name}: {before:,} -> {after:,} rows ({before-after:,} removed)")

def fingerprint(df):
    data = pd.util.hash_pandas_object(
        df.sort_index(axis=1).fillna("<NA>"),
        index=True
    ).values.tobytes()
    return hashlib.sha256(data).hexdigest()[:16]


## 3. Expected Raw Files

The first version of the pipeline expects cached/raw files:

- `data/raw/sec_form4.csv`
- `data/raw/market_data.csv`
- `data/raw/news.csv`

Later ingestion agents can populate these automatically. Keeping ingestion separate from preprocessing makes repeated runs easier to reproduce.


In [3]:
RAW_FILES = {
    "sec": PATHS["raw"] / "sec_form4.csv",
    "market": PATHS["raw"] / "market_data.csv",
    "news": PATHS["raw"] / "news.csv",
}

for name, path in RAW_FILES.items():
    print(f"{name}: {path} | exists={path.exists()}")


sec: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\sec_form4.csv | exists=True
market: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\market_data.csv | exists=True
news: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\news.csv | exists=True


# 4. SEC Form 4 Preprocessing

In [4]:
def preprocess_sec(df):
    before = len(df)
    df = normalize_column_names(df)

    required = ["ticker", "transaction_date", "owner_name"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"SEC data missing required columns: {missing}")

    optional = [
        "company_name", "owner_title", "filing_date",
        "transaction_code", "acquired_disposed",
        "shares", "price", "accession_number", "source_url"
    ]
    for col in optional:
        if col not in df.columns:
            df[col] = pd.NA

    df["ticker"] = normalize_ticker(df["ticker"])
    for col in ["owner_name", "company_name", "owner_title"]:
        df[col] = clean_text(df[col])

    df["transaction_date"] = to_utc(df["transaction_date"])
    df["filing_date"] = to_utc(df["filing_date"])

    df["shares"] = pd.to_numeric(df["shares"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["transaction_value"] = df["shares"] * df["price"]

    df["transaction_code"] = df["transaction_code"].astype("string").str.strip().str.upper()
    df["acquired_disposed"] = df["acquired_disposed"].astype("string").str.strip().str.upper()

    df = df.dropna(subset=["ticker", "transaction_date", "owner_name"])
    df = df.drop_duplicates(
        subset=["accession_number", "ticker", "owner_name", "transaction_date",
                "transaction_code", "shares", "price"]
    )

    df = df.sort_values(["ticker", "transaction_date"]).reset_index(drop=True)
    report_rows("SEC Form 4", before, len(df))
    return df


# 5. Market Data Preprocessing

In [5]:
def preprocess_market(df):
    before = len(df)
    df = normalize_column_names(df)

    required = ["ticker", "date", "close"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Market data missing required columns: {missing}")

    for col in ["open", "high", "low", "adj_close", "volume"]:
        if col not in df.columns:
            df[col] = pd.NA

    df["ticker"] = normalize_ticker(df["ticker"])
    df["date"] = to_utc(df["date"])

    for col in ["open", "high", "low", "close", "adj_close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["ticker", "date", "close"])
    df = df.drop_duplicates(subset=["ticker", "date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    df["daily_return"] = df.groupby("ticker")["close"].pct_change()
    df["volume_change"] = df.groupby("ticker")["volume"].pct_change()

    if "adj_close" in df.columns and df["adj_close"].isna().all():
        df = df.drop(columns=["adj_close"])

    report_rows("Market", before, len(df))
    return df


# 6. Financial News Preprocessing

In [6]:
COMPANY_ALIASES = {
    "AAPL": [
        "apple",
        "aapl",
    ],
}


def safe_lower(value):
    if pd.isna(value):
        return ""

    return str(value).lower()


def get_news_relevance_score(row):
    ticker = str(row.get("ticker", "")).upper()

    aliases = COMPANY_ALIASES.get(
        ticker,
        [ticker.lower()],
    )

    title = safe_lower(row.get("title"))

    article_text = " ".join(
        [
            title,
            safe_lower(row.get("description")),
            safe_lower(row.get("content")),
        ]
    )

    if any(alias in title for alias in aliases):
        return 2

    if any(alias in article_text for alias in aliases):
        return 1

    return 0


In [7]:
def preprocess_news(df):
    before = len(df)
    df = normalize_column_names(df)

    required = ["published_at", "title"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"News data missing required columns: {missing}")

    optional = [
        "ticker",
        "company_name",
        "description",
        "content",
        "source",
        "url",
    ]
    for col in optional:
        if col not in df.columns:
            df[col] = pd.NA

    df["ticker"] = normalize_ticker(df["ticker"])
    df["published_at"] = to_utc(df["published_at"])

    for col in [
        "title",
        "description",
        "content",
        "company_name",
        "source",
    ]:
        df[col] = clean_text(df[col])

    with_url = df[df["url"].notna()].drop_duplicates(subset=["url"])
    without_url = df[df["url"].isna()].drop_duplicates(
        subset=["title", "published_at"]
    )
    df = pd.concat(
        [with_url, without_url],
        ignore_index=True,
    )

    df = df.dropna(
        subset=[
            "published_at",
            "title",
        ]
    )

    df["text"] = (
        df["title"].fillna("")
        + ". "
        + df["description"].fillna("")
        + " "
        + df["content"].fillna("")
    )
    df["text"] = clean_text(df["text"])

    before_relevance = len(df)

    df["relevance_score"] = df.apply(
        get_news_relevance_score,
        axis=1,
    )

    print("\nNews relevance scores before filtering:")
    print(
        df["relevance_score"]
        .value_counts()
        .sort_index()
    )

    df = df[
        df["relevance_score"] >= 1
    ].copy()

    print(
        "News relevance filter:",
        f"{before_relevance:,} -> {len(df):,} rows "
        f"({before_relevance - len(df):,} removed)",
    )

    df = (
        df.sort_values(
            "published_at",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    report_rows(
        "News",
        before,
        len(df),
    )

    return df


# 7. Run Available Preprocessing

In [8]:
processed = {}

if RAW_FILES["sec"].exists():
    processed["sec"] = preprocess_sec(pd.read_csv(RAW_FILES["sec"]))
    display(processed["sec"].head())
else:
    print("SEC raw file not found yet.")

if RAW_FILES["market"].exists():
    processed["market"] = preprocess_market(pd.read_csv(RAW_FILES["market"]))
    display(processed["market"].head())
else:
    print("Market raw file not found yet.")

if RAW_FILES["news"].exists():
    processed["news"] = preprocess_news(pd.read_csv(RAW_FILES["news"]))
    display(processed["news"].head())
else:
    print("News raw file not found yet.")


SEC Form 4: 61 -> 57 rows (4 removed)


,ticker,company_name,owner_name,owner_title,transaction_date,filing_date,transaction_code,acquired_disposed,shares,price,accession_number,source_url,transaction_value
0,AAPL,Apple Inc.,WAGNER SUSAN,Director,2026-02-24 00:00:00+00:00,2026-02-26 00:00:00+00:00,A,A,1139.0,0.00,0001059235-26-000004,https://www.sec.gov/Archives/edgar/data/320193/000105923526000004/wk-form4_1772148856.xml,0.00
1,AAPL,Apple Inc.,Newstead Jennifer,"SVP, GC and Secretary",2026-03-15 00:00:00+00:00,2026-03-17 00:00:00+00:00,M,A,60208.0,NaN,0001780525-26-000005,https://www.sec.gov/Archives/edgar/data/320193/000178052526000005/wk-form4_1773786674.xml,NaN
2,AAPL,Apple Inc.,Newstead Jennifer,"SVP, GC and Secretary",2026-03-15 00:00:00+00:00,2026-03-17 00:00:00+00:00,F,D,32528.0,250.12,0001780525-26-000005,https://www.sec.gov/Archives/edgar/data/320193/000178052526000005/wk-form4_1773786674.xml,8135903.36
3,AAPL,Apple Inc.,O'BRIEN DEIRDRE,Senior Vice President,2026-04-01 00:00:00+00:00,2026-04-03 00:00:00+00:00,M,A,64317.0,NaN,0001140361-26-013192,https://www.sec.gov/Archives/edgar/data/320193/000114036126013192/form4.xml,NaN
4,AAPL,Apple Inc.,O'BRIEN DEIRDRE,Senior Vice President,2026-04-01 00:00:00+00:00,2026-04-03 00:00:00+00:00,F,D,34315.0,255.63,0001140361-26-013192,https://www.sec.gov/Archives/edgar/data/320193/000114036126013192/form4.xml,8771943.45


Market: 100 -> 100 rows (0 removed)


,ticker,date,open,high,low,close,volume,daily_return,volume_change
0,AAPL,2026-04-28 00:00:00+00:00,272.335,273.23,268.6600,270.71,40018940,NaN,NaN
1,AAPL,2026-04-29 00:00:00+00:00,267.550,271.04,267.0400,270.17,30047869,-0.001995,-0.249159
2,AAPL,2026-04-30 00:00:00+00:00,270.500,276.00,268.1400,271.35,91848230,0.004368,2.056730
3,AAPL,2026-05-01 00:00:00+00:00,278.855,287.22,278.3700,280.14,79915442,0.032394,-0.129919
4,AAPL,2026-05-04 00:00:00+00:00,279.655,280.63,274.8601,276.83,46668401,-0.011816,-0.416028



News relevance scores before filtering:
relevance_score
0    24
1     8
2    16
Name: count, dtype: int64
News relevance filter: 48 -> 24 rows (24 removed)
News: 50 -> 24 rows (26 removed)


,ticker,published_at,title,description,content,source,url,company_name,text,relevance_score
0,AAPL,2026-09-20 17:05:00+00:00,Wall Street Brunch: U.S.-China Summit In Spotlight (NYSEARCA:SPY),"Trump and Xi meet with trade, tariffs and AI on deck. Trump pitches an AI Force while Google reveals a Gemini breach...","Getty Images Download this episode on Apple Podcasts/Spotify or listen below: Trump and Xi meet with trade, tariffs ...",seekingalpha.com,https://seekingalpha.com/article/4948147-wall-street-brunch-us-china-summit-in-spotlight,<NA>,"Wall Street Brunch: U.S.-China Summit In Spotlight (NYSEARCA:SPY). Trump and Xi meet with trade, tariffs and AI on d...",1
1,AAPL,2026-09-20 16:37:00+00:00,"Prediction: Even With the $1,999 Price Tag, Apple's Foldable iPhone Duo Will Be a Hit and Apple Will Join Nvidia in ...",Apple is on track to do something great that it hasn't done since 2012.,"Apple's (NASDAQ: AAPL) new iPhones are here, and this could be just the ticket to get the consumer tech titan back o...",finance.yahoo.com,https://finance.yahoo.com/markets/stocks/articles/prediction-even-1-999-price-163700612.html,<NA>,"Prediction: Even With the $1,999 Price Tag, Apple's Foldable iPhone Duo Will Be a Hit and Apple Will Join Nvidia in ...",2
2,AAPL,2026-09-20 16:25:40+00:00,AAPL Looks 17.3% Overvalued on GF Value™ as Insider Selling Persists,"On September 20, 2026, Australian Prime Minister Anthony Albanese revealed insights from a meeting with Apple Inc. (...","On September 20, 2026, Australian Prime Minister Anthony Albanese revealed insights from a meeting with Apple Inc. (...",gurufocus.com,https://www.gurufocus.com/news/9089254/aapl-looks-173-overvalued-on-gf-value-as-insider-selling-persists,<NA>,"AAPL Looks 17.3% Overvalued on GF Value™ as Insider Selling Persists. On September 20, 2026, Australian Prime Minist...",2
3,AAPL,2026-09-20 11:30:29+00:00,Apple Enhances Durability with New A20 Pro Chip and Ceramic Shield in iPhone 18 Pro,"On September 20, 2026, Apple's advancements in product durability are making headlines with the launch of its new A2...","On September 20, 2026, Apple's advancements in product durability are making headlines with the launch of its new A2...",gurufocus.com,https://www.gurufocus.com/news/9089235/apple-enhances-durability-with-new-a20-pro-chip-and-ceramic-shield-in-iphone-...,<NA>,"Apple Enhances Durability with New A20 Pro Chip and Ceramic Shield in iPhone 18 Pro. On September 20, 2026, Apple's ...",2
4,AAPL,2026-09-19 12:50:00+00:00,John Ternus's First iPhone Launch Prompted a Bank of America Price Target Cut. Is Apple Stock a Buy?,Apple stock just got a price target cut due to the memory processor crunch.,"John Ternus recently hosted his first iPhone event as Apple (NASDAQ: AAPL) CEO, and following the debut of the new i...",finance.yahoo.com,https://finance.yahoo.com/markets/stocks/articles/john-ternuss-first-iphone-launch-125000640.html,<NA>,John Ternus's First iPhone Launch Prompted a Bank of America Price Target Cut. Is Apple Stock a Buy?. Apple stock ju...,2


## 8. News Relevance Validation

Financial-news APIs can return articles where the target ticker is only weakly associated with the story. A lightweight relevance rule is applied before news is passed to downstream agents.

- **2** — company/ticker appears in the title
- **1** — company/ticker appears in the description or content
- **0** — no direct company reference

Articles with a score of 0 are removed. The rule is intentionally simple and transparent so that more contextual relevance decisions can be handled later by the news-analysis agents.


In [9]:
if "news" in processed:
    news = processed["news"]

    print("Clean news rows:", len(news))
    print("Missing titles:", news["title"].isna().sum())
    print(
        "Missing publication dates:",
        news["published_at"].isna().sum(),
    )
    print(
        "Duplicate URLs:",
        news["url"]
        .dropna()
        .duplicated()
        .sum(),
    )
    print(
        "Average relevance score:",
        round(
            news["relevance_score"].mean(),
            2,
        ),
    )

    display(
        news["relevance_score"]
        .value_counts()
        .sort_index()
        .rename_axis("relevance_score")
        .reset_index(name="article_count")
    )


Clean news rows: 24
Missing titles: 0
Missing publication dates: 0
Duplicate URLs: 0
Average relevance score: 1.67


,relevance_score,article_count
0,1,8
1,2,16


# 9. Data Quality Summary

In [10]:
quality = []

for name, frame in processed.items():
    quality.append({
        "dataset": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
        "fingerprint": fingerprint(frame),
    })

quality_df = pd.DataFrame(quality)
display(quality_df) if len(quality_df) else print("No processed datasets available yet.")


,dataset,rows,columns,duplicate_rows,missing_cells,fingerprint
0,sec,57,13,0,48,2a9558af6146e008
1,market,100,9,0,2,4caa85a9db2f5407
2,news,24,10,0,24,bdd048c5257a03be


# 10. Save Standardized Outputs

In [11]:
OUTPUT_FILES = {
    "sec": PATHS["processed"] / "sec_form4_clean.csv",
    "market": PATHS["processed"] / "market_data_clean.csv",
    "news": PATHS["processed"] / "news_clean.csv",
}

for name, frame in processed.items():
    frame.to_csv(OUTPUT_FILES[name], index=False)
    print("Saved:", OUTPUT_FILES[name])


Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\sec_form4_clean.csv
Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\market_data_clean.csv
Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\processed\news_clean.csv


# 11. Preprocessing Manifest

In [12]:
manifest = {
    "seed": SEED,
    "datasets": {
        name: {
            "rows": len(frame),
            "columns": list(frame.columns),
            "fingerprint": fingerprint(frame),
            "output_file": str(OUTPUT_FILES[name]),
        }
        for name, frame in processed.items()
    }
}

manifest_path = PATHS["processed"] / "preprocessing_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))


{
  "seed": 42,
  "datasets": {
    "sec": {
      "rows": 57,
      "columns": [
        "ticker",
        "company_name",
        "owner_name",
        "owner_title",
        "transaction_date",
        "filing_date",
        "transaction_code",
        "acquired_disposed",
        "shares",
        "price",
        "accession_number",
        "source_url",
        "transaction_value"
      ],
      "fingerprint": "2a9558af6146e008",
      "output_file": "C:\\Users\\admin\\Desktop\\AAI520\\Investment-Research-Multi-Agent-System\\data\\processed\\sec_form4_clean.csv"
    },
    "market": {
      "rows": 100,
      "columns": [
        "ticker",
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "daily_return",
        "volume_change"
      ],
      "fingerprint": "4caa85a9db2f5407",
      "output_file": "C:\\Users\\admin\\Desktop\\AAI520\\Investment-Research-Multi-Agent-System\\data\\processed\\market_data_clean.csv"
    },
    "n

# 12. Next Step: News Research Pipeline

After preprocessing, the cleaned and relevance-filtered news is ready for the prompt-chaining workflow:

**ingest → preprocess → classify → extract → summarize**

The next notebook, `03_news_processing_chain.ipynb`, will:

- classify each relevant article into a financial-news category;
- extract structured facts and entities;
- generate an evidence-grounded research summary;
- save a standardized handoff for the Planner and Router agents.

The preprocessing layer does not make investment recommendations. Its purpose is to provide clean, relevant, traceable evidence to the downstream agent system.


## Completion Checklist

- [x] Confirm SEC ingestion schema.
- [x] Confirm market-data schema.
- [x] Confirm news schema.
- [x] Normalize columns and timestamps.
- [x] Remove duplicate records.
- [x] Create standardized news text.
- [x] Apply basic news relevance filtering.
- [x] Report preprocessing row counts.
- [x] Run data-quality checks.
- [x] Save cleaned outputs to `data/processed/`.
- [x] Generate `preprocessing_manifest.json`.
- [ ] Build `03_news_processing_chain.ipynb`.
